In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from steering_vectors import train_steering_vector, record_activations

In [2]:
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",   
    cache_dir='../cache',
    torch_dtype=torch.float16,
    device_map='balanced',
)

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    cache_dir='../cache',
    torch_dtype=torch.float16,
    device_map='balanced',
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.32s/it]


In [5]:
generation_config = GenerationConfig(max_new_tokens=40)

In [66]:
tokenizer.pad_token = tokenizer.eos_token

In [101]:
with record_activations(model, layer_type='self_attn') as records:
    prompt = "Question: Is it true that crystals have magic healing properties? (A) Yes (B) No. Response: (B)"
    inputs = tokenizer([prompt], return_tensors="pt", padding=True)
    for k, v in inputs.items():
        inputs[k] = v.to('mps')
        
    outputs = model.forward(**inputs)

In [102]:
inputs['input_ids']

tensor([[    1,   894, 29901,  1317,   372,  1565,   393, 10901,   303,  1338,
           505, 15709,   540, 12818,  4426, 29973,   313, 29909, 29897,  3869,
           313, 29933, 29897,  1939, 29889, 13291, 29901,   313, 29933, 29897]],
       device='mps:0')

In [81]:
records[5][0].shape

torch.Size([2, 47, 4096])

In [68]:
records[5][0].shape

torch.Size([2, 47, 4096])

In [49]:
inputs

{'input_ids': tensor([[    1,  1317,   372,  1565,   393, 10901,   303,  1338,   505, 15709,
           540, 12818,  4426, 29973,   319, 29897,  3869,   350, 29897,  1939,
         29889,   673, 29901,   319]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='mps:0')}

In [47]:
inputs['input_ids'].shape

torch.Size([1, 24])

In [33]:
outputs.logits.shape

torch.Size([1, 24, 32000])

In [37]:
records[5][0].shape

torch.Size([1, 24, 4096])

In [10]:
records[5]

40

In [19]:
outputs.shape[1] - inputs['input_ids'].shape[1]

40

In [3]:
# training samples are tuples of (positive_prompt, negative_prompt)
training_samples = [
    (
        "The capital of England is London",
        "The capital of England is Beijing"
    ),
    (
        "The capital of France is Paris",
        "The capital of France is Berlin"
    )
    # ...
]

steering_vector = train_steering_vector(
    model,
    tokenizer,
    training_samples,
    show_progress=True,
    # layers=[1,2,3],
)

Training steering vector: 100%|██████████| 2/2 [00:00<00:00,  2.55it/s]


In [24]:
steering_vector.layer_activations[5].dtype

torch.float16

In [17]:
with steering_vector.apply(model, multiplier=0.1):

    prompt = "Is it true that crystals have magic healing properties?"
    inputs = tokenizer(prompt, return_tensors="pt")
    for k, v in inputs.items():
        inputs[k] = v.to('mps')
    outputs = model.generate(**inputs, generation_config=GenerationConfig(max_new_tokens=40))

In [18]:
outputs

tensor([[    1,  1317,   372,  1565,   393, 10901,   303,  1338,   505, 15709,
           540, 12818,  4426, 29973,    13,  8439,   526,  1784,  2305,  1058,
          4658,   393,   278,   540, 12818, 10801,   310,   278,  5199,  3573,
           526,   427, 29308,   491,   278, 10122,   310,  3058,  4072,   310,
         10901,   303,   856,    13,  5618,   338,   278,  6593,   310,   278,
          1734,  5129, 29888, 10904]], device='mps:0')

In [19]:
tokenizer.decode(token_ids=outputs[0])

'<s> Is it true that crystals have magic healing properties?\nThere are many people who believe that the healing powers of the human body are enhanced by the presence of certain types of cryst...\nWhat is the meaning of the word ‘fashion'

In [19]:
outputs

tensor([[ 3792,   340,  2081,   326, 24770,   423,  5536, 11516,  6608,    30,
          4188,  4188,  4188,  4188,  4188,  4188,  4188,  4188,  4188,  4188,
          4188,  4188,  4188,  4188,  4188,  4188,  4188,  4188,  4188,  4188]],
       device='mps:0')